# Plots — LHS 6050 b

Loads the posterior samples written by `run_exomdn_lhs6050b.ipynb` and draws, for both
mass fractions and radius fractions: a four-panel ridgeplot (core / mantle / water /
atmosphere) and the standard ExoMDN cornerplot. Run `run_exomdn_lhs6050b.ipynb` first.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import gaussian_kde


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / "exomdn" / "mdn_model.py").is_file():
            return parent
    raise FileNotFoundError("Could not find ExoMDN repo root (no exomdn/mdn_model.py upward)")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from exomdn.plotting import cornerplot

## Ridgeplot function

Four panels, one per interior layer. Each panel shows the posterior KDE for that layer's
fraction, with vertical markers at the 5th, 50th and 95th percentiles. The atmosphere
panel uses a log x-axis (floor 1e-3) because its posterior bunches near zero with a long
tail; a linear axis would hide the structure.

In [ ]:
LAYERS_MF = ["core_mf", "mantle_mf", "water_mf", "atmosphere_mf"]
LAYERS_RF = ["core_rf", "mantle_rf", "water_rf", "atmosphere_rf"]
LAYER_TITLES = ["Core", "Mantle", "Water", "Gas"]
LAYER_COLORS = {
    "core": "#0C0920",
    "mantle": "#4F4E7E",
    "water": "#64A2CE",
    "atmosphere": "#A1A6B4",
}
XLABEL_MF = [
    r"$M_\mathrm{core}/M_\mathrm{planet}$",
    r"$M_\mathrm{mantle}/M_\mathrm{planet}$",
    r"$M_\mathrm{water}/M_\mathrm{planet}$",
    r"$M_\mathrm{gas}/M_\mathrm{planet}$",
]
XLABEL_RF = [
    r"$R_\mathrm{core}/R_\mathrm{planet}$",
    r"$R_\mathrm{mantle}/R_\mathrm{planet}$",
    r"$R_\mathrm{water}/R_\mathrm{planet}$",
    r"$R_\mathrm{gas}/R_\mathrm{planet}$",
]


def kde_1d(x, grid, subsample=3000):
    """Gaussian KDE on a 0-1 grid, subsampling large arrays for speed."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 10:
        return np.zeros_like(grid)
    x = np.clip(x, 0.0, 1.0)
    if len(x) > subsample:
        rng = np.random.default_rng(seed=42)
        x = rng.choice(x, size=subsample, replace=False)
    return gaussian_kde(x)(grid)


def plot_planet_ridge(samples, mode="mf", *, planet_name=None,
                      atm_xlim=(1e-3, 0.5), fig_width=None, fig_height=3.0):
    """Four-panel posterior ridgeplot (core / mantle / water / atmosphere).

    samples : DataFrame from Model.predict_with_error (first output).
    mode    : 'mf' for mass fractions or 'rf' for radius fractions.
    """
    if mode not in ("mf", "rf"):
        raise ValueError(f"mode must be 'mf' or 'rf', got {mode!r}")
    layers = LAYERS_MF if mode == "mf" else LAYERS_RF
    xlabels = XLABEL_MF if mode == "mf" else XLABEL_RF

    sns.set_theme(style="ticks", font_scale=1.1)
    w = fig_width if fig_width is not None else 11.0
    col_w_ratios = [1.0, 1.0, 1.0, 0.7] if mode == "mf" else [1.0, 1.0, 1.0, 1.0]

    fig, axes = plt.subplots(
        1, 4, figsize=(w, fig_height),
        gridspec_kw={"width_ratios": col_w_ratios, "wspace": 0.15},
    )

    grid = np.linspace(0.0, 1.0, 1000)
    quantiles = (0.05, 0.5, 0.95)

    for ax, layer, xlabel, title in zip(axes, layers, xlabels, LAYER_TITLES):
        x = samples[layer].to_numpy()
        kde = kde_1d(x, grid)
        peak = kde.max()
        kde_norm = kde / peak if peak > 0 else kde

        color = LAYER_COLORS[layer.split("_")[0]]
        ax.fill_between(grid, 0.0, kde_norm,
                        color=color, alpha=0.5, linewidth=0, zorder=1)
        ax.plot(grid, kde_norm, color="black", linewidth=2.0, zorder=2)

        # Median + 5/95 percentile markers along the baseline
        #q05, q50, q95 = np.quantile(x[np.isfinite(x)], quantiles)
        #for q, ls in [(q05, ":"), (q50, "-"), (q95, ":")]:
        #    ax.axvline(q, color=color, linestyle=ls, linewidth=1.0, alpha=0.7, zorder=0)

        xlim = atm_xlim if (mode == "mf" and layer == "atmosphere_mf") else (0.0, 1.0)
        ax.set_xlim(*xlim)
        if layer in ("atmosphere_mf", "atmosphere_rf"):
            # Atmosphere posteriors bunch near zero; log axis with 1e-3 floor.
            # Label only 1e-3 and 1e-1 (skip 1e-2).
            ax.set_xscale("log")
            ax.set_xlim(1e-3, xlim[1])
            ax.set_xticks([1e-3, 1e-1])
            ax.minorticks_off()
            # Left-align the 1e-3 label so it clears the previous panel's 1.0.
            tick_texts = ax.set_xticklabels([r"$10^{-3}$", r"$10^{-1}$"])
            tick_texts[0].set_horizontalalignment("left")
        elif layer.split("_")[0] in ("mantle", "water"):
            # Drop the leading 0.0 so it doesn't crowd the previous panel.
            ax.set_xticks([0.5, 1.0])
        else:
            ax.set_xticks([0.0, 0.5, 1.0])
        ax.set_ylim(-0.05, 1.15)
        ax.set_yticks([])
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_title(title, fontsize=12, fontweight="bold")
        sns.despine(ax=ax, left=True)

    if planet_name:
        fig.suptitle(planet_name, fontsize=12, fontweight="bold", y=1.0)
    fig.tight_layout()
    return fig

## Load the samples

In [ ]:
samples = pd.read_parquet(Path.cwd() / "lhs6050b_samples.parquet")
print(f"loaded {len(samples):,} posterior samples")

## Mass-fraction ridgeplot

In [ ]:
fig_mf = plot_planet_ridge(samples, mode="mf")#, planet_name="LHS 6050 b")
fig_mf.savefig("ridge_mass_fractions_lhs6050b.pdf", bbox_inches="tight")
fig_mf.savefig("ridge_mass_fractions_lhs6050b.png", dpi=200, bbox_inches="tight")

## Radius-fraction ridgeplot

In [ ]:
fig_rf = plot_planet_ridge(samples, mode="rf")#, planet_name="LHS 6050 b")
fig_rf.savefig("ridge_radius_fractions_lhs6050b.pdf", bbox_inches="tight")
fig_rf.savefig("ridge_radius_fractions_lhs6050b.png", dpi=200, bbox_inches="tight")

## Mass-fraction cornerplot

In [ ]:
g_mf = cornerplot(samples, LAYERS_MF, height=2)
g_mf.figure.savefig("corner_mass_fractions_lhs6050b.pdf", bbox_inches="tight")
g_mf.figure.savefig("corner_mass_fractions_lhs6050b.png", dpi=200, bbox_inches="tight")

## Radius-fraction cornerplot

In [ ]:
g_rf = cornerplot(samples, LAYERS_RF, height=2)
g_rf.figure.savefig("corner_radius_fractions_lhs6050b.pdf", bbox_inches="tight")
g_rf.figure.savefig("corner_radius_fractions_lhs6050b.png", dpi=200, bbox_inches="tight")